# 🎮 エージェント対戦とナッシュレーティング評価ノートブック

このノートブックでは、複数の学習済みエージェントを対戦させて対戦ログを記録し、ナッシュ均衡に基づいたレーティングを算出します。

In [1]:
# ✅ 初期設定：必要モジュールのインポート
import os
import json
from train import AgentFactory, Env_Geister
from geister_game import GeisterGame

# ログ保存先
LOG_DIR = "./logs"
os.makedirs(LOG_DIR, exist_ok=True)

In [3]:
# ✅ 1. エージェント構成・重みの読み込み
agent_info_list = [
    {"id": "Agent1", "config": "models_geister_agentA/eps1200/config.json", "weights": "models_geister_agentA/eps1200/weights.pth"},
    {"id": "Agent2", "config": "models_geister_agentA/eps1000/config.json", "weights": "models_geister_agentA/eps1000/weights.pth"},
    {"id": "Agent3", "config": "models_geister_agentA/eps800/config.json", "weights": "models_geister_agentA/eps800/weights.pth"},
    {"id": "Agent4", "config": "models_geister_agentA/eps600/config.json", "weights": "models_geister_agentA/eps600/weights.pth"},
    {"id": "Agent5", "config": "models_geister_agentA/eps400/config.json", "weights": "models_geister_agentA/eps400/weights.pth"},
    {"id": "Agent6", "config": "models_geister_agentA/eps200/config.json", "weights": "models_geister_agentA/eps200/weights.pth"},
    {"id": "Agent7", "config": "models_geister_agentA/eps100/config.json", "weights": "models_geister_agentA/eps100/weights.pth"},
]

agents = {}
for info in agent_info_list:
    with open(info["config"], "r") as f:
        cfg = json.load(f)
    agent = AgentFactory.create_cqc_agent("A", GeisterGame(board_size=4, num_ghosts_per_player=2), cfg, info["weights"])
    agents[info["id"]] = agent

In [4]:
# ✅ 2. 総当たり対戦ログの生成（階層ディレクトリ）
for id_a, agent_a in agents.items():
    for id_b, agent_b in agents.items():
        if id_a == id_b:
            continue
        game = GeisterGame(board_size=4, num_ghosts_per_player=2)
        agent_a.player_id, agent_b.player_id = "A", "B"
        agent_a.game, agent_b.game = game, game
        env = Env_Geister(agent_a, agent_b, game)
        winner, moves, board = env.play_one_game_with_log()
        log = {
            "model_A": {"id": id_a},
            "model_B": {"id": id_b},
            "result": {"winner": "model_A" if winner == "A" else "model_B" if winner == "B" else "Draw"},
            "moves": moves
        }
        save_dir = os.path.join(LOG_DIR, f"{id_a}_vs_{id_b}")
        os.makedirs(save_dir, exist_ok=True)
        match_index = len(os.listdir(save_dir))
        with open(os.path.join(save_dir, f"match_{match_index}.json"), "w") as f:
            json.dump(log, f, indent=2)

In [6]:
# ✅ 3. ナッシュレーティングの計算
!python compute_elo_rating.py

Traceback (most recent call last):
  File "c:\Users\kishu\Qugeister\compute_elo_rating.py", line 52, in <module>
    ]).sort_values(by="EloRating", ascending=False)
       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kishu\quantum-env\Lib\site-packages\pandas\core\frame.py", line 7196, in sort_values
    k = self._get_label_or_level_values(by[0], axis=axis)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\kishu\quantum-env\Lib\site-packages\pandas\core\generic.py", line 1911, in _get_label_or_level_values
    raise KeyError(key)
KeyError: 'EloRating'
